# Step 2 — Cross-Country 10y PCA (Global Factor)

Correlation-matrix PCA on daily changes across seven countries. Extracts the global level factor (PC1), reports loadings and each country's common vs idiosyncratic share, with the UK in focus.

**Script notes**

```
Step 2 - Cross-country 10-year PCA (the spine)
Question: how much of daily 10y yield co-movement is a common GLOBAL factor,
and how 'untethered' is the UK (its idiosyncratic / non-common share)?

Method:
  - daily CHANGES (bp), correlation matrix (standardised), eigendecomposition
  - PC1 sign fixed so loadings are positive => 'global level' factor
  - communality_k = sum over retained PCs of (loading^2)  (since standardised,
        loadings = corr(series, PC); communality = R^2 explained by retained PCs)
  - idiosyncratic share = 1 - communality
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import numpy as np
import pandas as pd

In [2]:
ch = pd.read_csv("./changes_bp.csv", index_col=0, parse_dates=True)

In [3]:
CTRY10 = ["US_10y", "UK_10y", "DE_10y", "FR_10y", "IT_10y", "JP_10y", "CA_10y"]
X = ch[CTRY10].copy()
labels = [c[:2] for c in CTRY10]

In [4]:
# ---- standardise (correlation-matrix PCA) ----
Z = (X - X.mean()) / X.std(ddof=1)
C = np.corrcoef(Z.values, rowvar=False)          # 7x7 correlation matrix

In [5]:
# ---- eigendecomposition ----
eigval, eigvec = np.linalg.eigh(C)               # ascending
order = np.argsort(eigval)[::-1]                  # descending
eigval, eigvec = eigval[order], eigvec[:, order]
var_explained = eigval / eigval.sum()

In [6]:
# loadings = eigvec * sqrt(eigval)  (corr of each series with each PC)
loadings = eigvec * np.sqrt(eigval)

In [7]:
# ---- sign convention: PC1 positive (yields rise together) ----
if (loadings[:, 0] < 0).sum() > len(labels) / 2:
    loadings[:, 0] *= -1; eigvec[:, 0] *= -1

In [8]:
L = pd.DataFrame(loadings, index=labels,
                 columns=[f"PC{i+1}" for i in range(len(labels))])

In [9]:
print("="*64)
print("CROSS-COUNTRY 10Y PCA  (correlation matrix of daily changes, bp)")
print(f"sample: {X.index.min().date()} -> {X.index.max().date()}   n={len(X)}")
print("="*64)

CROSS-COUNTRY 10Y PCA  (correlation matrix of daily changes, bp)
sample: 2024-01-05 -> 2026-05-29   n=519


In [10]:
print("\nVariance explained by each PC:")
for i, v in enumerate(var_explained):
    print(f"  PC{i+1}: {v*100:5.1f}%   (cumulative {var_explained[:i+1].sum()*100:5.1f}%)   eigenvalue={eigval[i]:.3f}")


Variance explained by each PC:
  PC1:  54.9%   (cumulative  54.9%)   eigenvalue=3.844
  PC2:  21.4%   (cumulative  76.3%)   eigenvalue=1.497
  PC3:  10.5%   (cumulative  86.8%)   eigenvalue=0.736
  PC4:   6.3%   (cumulative  93.1%)   eigenvalue=0.439
  PC5:   3.1%   (cumulative  96.2%)   eigenvalue=0.215
  PC6:   2.2%   (cumulative  98.4%)   eigenvalue=0.156
  PC7:   1.6%   (cumulative 100.0%)   eigenvalue=0.113


In [11]:
print("\nLoadings (correlation of each country's daily change with each PC):")
print(L.round(3).to_string())


Loadings (correlation of each country's daily change with each PC):
      PC1    PC2    PC3    PC4    PC5    PC6    PC7
US  0.778  0.460  0.210 -0.245 -0.062 -0.265  0.064
UK  0.847  0.254 -0.166  0.268 -0.333  0.090  0.009
DE  0.719 -0.607 -0.154 -0.170 -0.044 -0.064 -0.235
FR  0.691 -0.617 -0.236 -0.176  0.028  0.059  0.226
IT  0.818  0.208 -0.292  0.339  0.285 -0.078 -0.014
JP  0.497 -0.472  0.680  0.260  0.025  0.004  0.017
CA  0.781  0.453  0.195 -0.254  0.124  0.253 -0.053


In [12]:
# ---- communality from PC1 only, and from PC1+PC2 ----
comm1 = loadings[:, 0]**2
comm2 = (loadings[:, [0, 1]]**2).sum(axis=1)
tbl = pd.DataFrame({
    "PC1_loading": loadings[:, 0],
    "communality_PC1": comm1,
    "idiosyncratic_(1-PC1)": 1 - comm1,
    "communality_PC1+2": comm2,
    "idiosyncratic_(1-PC1&2)": 1 - comm2,
}, index=labels)
print("\nCommon vs idiosyncratic share of each country's daily-change variance:")
print(tbl.round(3).to_string())


Common vs idiosyncratic share of each country's daily-change variance:
    PC1_loading  communality_PC1  idiosyncratic_(1-PC1)  communality_PC1+2  idiosyncratic_(1-PC1&2)
US        0.778            0.606                  0.394              0.818                    0.182
UK        0.847            0.717                  0.283              0.781                    0.219
DE        0.719            0.518                  0.482              0.886                    0.114
FR        0.691            0.478                  0.522              0.858                    0.142
IT        0.818            0.669                  0.331              0.712                    0.288
JP        0.497            0.247                  0.753              0.470                    0.530
CA        0.781            0.611                  0.389              0.816                    0.184


In [13]:
# ---- cross-check: regress each country's standardised change on retained PC scores ----
scores = Z.values @ eigvec                        # observation scores on PCs
def r2_on_pcs(y_idx, k):
    y = Z.values[:, y_idx]
    P = scores[:, :k]
    beta, *_ = np.linalg.lstsq(P, y, rcond=None)
    yhat = P @ beta
    ss_res = ((y - yhat)**2).sum(); ss_tot = (y**2).sum()
    return 1 - ss_res/ss_tot
print("\nCross-check R^2 of regressing each country's change on retained PCs:")
for i, lab in enumerate(labels):
    print(f"  {lab}: R2(PC1)={r2_on_pcs(i,1):.3f}   R2(PC1-2)={r2_on_pcs(i,2):.3f}")


Cross-check R^2 of regressing each country's change on retained PCs:
  US: R2(PC1)=0.606   R2(PC1-2)=0.818
  UK: R2(PC1)=0.717   R2(PC1-2)=0.781
  DE: R2(PC1)=0.518   R2(PC1-2)=0.886
  FR: R2(PC1)=0.478   R2(PC1-2)=0.858
  IT: R2(PC1)=0.669   R2(PC1-2)=0.712
  JP: R2(PC1)=0.247   R2(PC1-2)=0.470
  CA: R2(PC1)=0.611   R2(PC1-2)=0.816


In [14]:
# ---- headline UK numbers ----
uk = labels.index("UK")
print("\n" + "-"*64)
print("UK FOCUS")
print(f"  UK PC1 loading           : {loadings[uk,0]:.3f}   (rank {sorted(loadings[:,0],reverse=True).index(loadings[uk,0])+1} of 7, higher=more global)")
print(f"  UK common share (PC1)    : {comm1[uk]*100:.1f}%")
print(f"  UK idiosyncratic (1-PC1) : {(1-comm1[uk])*100:.1f}%")
print(f"  cross-country avg idio   : {(1-comm1).mean()*100:.1f}%   (UK {'ABOVE' if (1-comm1[uk])>(1-comm1).mean() else 'BELOW'} average)")
print("-"*64)


----------------------------------------------------------------
UK FOCUS
  UK PC1 loading           : 0.847   (rank 1 of 7, higher=more global)
  UK common share (PC1)    : 71.7%
  UK idiosyncratic (1-PC1) : 28.3%
  cross-country avg idio   : 45.1%   (UK BELOW average)
----------------------------------------------------------------


In [15]:
L.to_csv("./pca10_loadings.csv")
tbl.to_csv("./pca10_communality.csv")
pd.Series(var_explained, index=L.columns, name="var_explained").to_csv("./pca10_varexplained.csv")
print("\nsaved pca10_loadings.csv, pca10_communality.csv, pca10_varexplained.csv")


saved pca10_loadings.csv, pca10_communality.csv, pca10_varexplained.csv
